# Project 03 — Credit Risk: PD / LGD / EAD & Expected Loss

## Notebook 01 — Data Preparation & Credit Risk Analysis

This notebook establishes the data and analytical foundation for the
credit-risk framework.

The notebook will:

- Introduce the fundamental components of credit risk
- Load and inspect the consumer-loan dataset
- Define the default target for Probability of Default (PD) modeling
- Assess data quality and missingness
- Explore default behavior across borrower and loan characteristics
- Engineer a limited number of credit-risk variables
- Segment borrowers by credit risk
- Create development and validation datasets
- Prepare a clean dataset for the PD, LGD, EAD, and Expected Loss models

## 1. Objective

The objective of this notebook is to prepare and understand the loan
portfolio that will be used throughout the credit-risk framework.

The analysis follows the fundamental credit-risk relationship:

$$
EL = PD \times LGD \times EAD
$$

where:

- **PD** — Probability of Default
- **LGD** — Loss Given Default
- **EAD** — Exposure at Default
- **EL** — Expected Loss

The notebook focuses primarily on understanding the portfolio and preparing
the data required to estimate these components in subsequent notebooks.

### Workflow

$$
\text{Raw Loan Data}
\rightarrow
\text{Data Quality}
\rightarrow
\text{Target Definition}
\rightarrow
\text{Feature Engineering}
\rightarrow
\text{Credit Risk Analysis}
\rightarrow
\text{Development / Validation Data}
$$

### Downstream Applications

The cleaned dataset produced here will be used by subsequent notebooks for:

* **Probability of Default (PD)**
* **Loss Given Default (LGD)**
* **Exposure at Default (EAD)**
* **Expected Loss**
* **Portfolio Credit Risk**
* **Stress Testing**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path


## 2. Dataset and Variable Selection

The source is the LendingClub loan-performance file `Loan_status_2007-2020Q3.gzip`. A focused set of 15 fields is loaded rather than the full source file.

The selected fields cover:

- **Outcome and timing:** `loan_status`, `issue_d`
- **Loan amount and pricing:** `loan_amnt`, `funded_amnt`, `int_rate`
- **Borrower affordability and credit profile:** `annual_inc`, `dti`, `fico_range_low`, `fico_range_high`, `emp_length`, `grade`
- **Borrower/loan characteristics:** `home_ownership`, `purpose`
- **Record and recovery information:** `id`, `recoveries`

`loan_status` is used to define the target, `issue_d` is used for the chronological split, and `id` is an identifier—not a PD predictor. `recoveries` is a post-outcome variable retained for later LGD work and must not be used as a PD predictor.

**Scope limitation:** this 15-field extract does not include loan term, full payment history, or outstanding principal at default. The later LGD/EAD work will therefore need to use clearly stated simplifications rather than claim a production-grade estimate.

In [3]:


data_path = Path("../data/raw/Loan_status_2007-2020Q3.gzip")

selected_columns = [
    "id",
    "issue_d",
    "loan_status",
    "loan_amnt",
    "funded_amnt",
    "int_rate",
    "annual_inc",
    "dti",
    "fico_range_low",
    "fico_range_high",
    "emp_length",
    "grade",
    "home_ownership",
    "purpose",
    "recoveries"
]

df = pd.read_csv(data_path, usecols=selected_columns)

df.shape

C:\Users\hp\AppData\Local\Temp\ipykernel_21020\478976603.py:21: DtypeWarning: Columns (1: id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, usecols=selected_columns)


(2925493, 15)

In [4]:
df.head()

,id,loan_amnt,funded_amnt,int_rate,grade,emp_length,home_ownership,annual_inc,issue_d,loan_status,purpose,dti,fico_range_low,fico_range_high,recoveries
0,1077501,5000.0,5000.0,10.65%,B,10+ years,RENT,24000.0,Dec-2011,Fully Paid,credit_card,27.65,735.0,739.0,0.0
1,1077430,2500.0,2500.0,15.27%,C,< 1 year,RENT,30000.0,Dec-2011,Charged Off,car,1.00,740.0,744.0,122.9
2,1077175,2400.0,2400.0,15.96%,C,10+ years,RENT,12252.0,Dec-2011,Fully Paid,small_business,8.72,735.0,739.0,0.0
3,1076863,10000.0,10000.0,13.49%,C,10+ years,RENT,49200.0,Dec-2011,Fully Paid,other,20.00,690.0,694.0,0.0
4,1075358,3000.0,3000.0,12.69%,B,1 year,RENT,80000.0,Dec-2011,Fully Paid,other,17.94,695.0,699.0,0.0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2925493 entries, 0 to 2925492
Data columns (total 15 columns):
 #   Column           Dtype  
---  ------           -----  
 0   id               object 
 1   loan_amnt        float64
 2   funded_amnt      float64
 3   int_rate         str    
 4   grade            str    
 5   emp_length       str    
 6   home_ownership   str    
 7   annual_inc       float64
 8   issue_d          str    
 9   loan_status      str    
 10  purpose          str    
 11  dti              float64
 12  fico_range_low   float64
 13  fico_range_high  float64
 14  recoveries       float64
dtypes: float64(7), object(1), str(7)
memory usage: 334.8+ MB


## 2. Data Quality Assessment

Before performing credit-risk analysis, the dataset is assessed for
missing values, duplicate observations, data types, and potential
invalid values.

The objective is to identify data-quality issues before feature
engineering and model development.

In [6]:
missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
)

missing_summary["missing_pct"] = missing_summary["missing_count"] / len(df) * 100

missing_summary.sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
emp_length,205221,7.014920
dti,3109,0.106273
annual_inc,5,0.000171
grade,1,0.000034
int_rate,1,0.000034
funded_amnt,1,0.000034
loan_amnt,1,0.000034
purpose,1,0.000034
home_ownership,1,0.000034
loan_status,1,0.000034


**Missingness finding.** Employment length is the main missing-data issue in the full extract (about 7.01% of rows). Missing DTI is much less common (about 0.11%), while missingness in the other selected fields is negligible. We will reassess missingness after restricting the data to terminal outcomes because filtering changes the modelling sample.

In [7]:
df.duplicated().sum()

np.int64(0)

**Duplicate check.** The check found **zero exact duplicate rows** across the selected fields. This check detects exact row duplicates; it does not imply that all loans have unique values for every borrower or loan characteristic.

In [8]:
df["loan_status"].value_counts()

loan_status
Fully Paid                                             1497783
Current                                                1031016
Charged Off                                             362548
Late (31-120 days)                                       16154
In Grace Period                                          10028
Late (16-30 days)                                         2719
Issued                                                    2062
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                    433
Name: count, dtype: int64

**Status review.** The source contains terminal outcomes (`Fully Paid`, `Charged Off`), active or unresolved statuses (`Current`, `Late`, `In Grace Period`, `Issued`), and a small `Default` category. For this project, the main PD target uses terminal outcomes only. That avoids treating every non-`Fully Paid` record as a confirmed default and mixing intermediate statuses with final outcomes.

## 4. Define the Terminal-Outcome Target and Prepare Fields

The modelling sample includes loans with clear terminal outcomes:

- `Fully Paid` → `default_flag = 0`
- `Charged Off` → `default_flag = 1`
- The two legacy *Does not meet the credit policy* variants are mapped according to their terminal status.

Rows with unresolved statuses—including `Current`, `Late`, `In Grace Period`, `Issued`, and the separate `Default` status—are excluded from this primary target definition. This is a deliberate simplification for a clean terminal-outcome classification problem.

In [9]:
terminal_statuses = [
    "Fully Paid",
    "Charged Off",
    "Does not meet the credit policy. Status:Fully Paid",
    "Does not meet the credit policy. Status:Charged Off",
]

status_to_default = {
    "Fully Paid": 0,
    "Charged Off": 1,
    "Does not meet the credit policy. Status:Fully Paid": 0,
    "Does not meet the credit policy. Status:Charged Off": 1,
}

# Keep only terminal outcomes, then map each outcome to a binary target.
df_model = df.loc[df["loan_status"].isin(terminal_statuses)].copy()
df_model["default_flag"] = df_model["loan_status"].map(status_to_default).astype("int8")

# Guard against unexpected statuses entering the selected sample.
assert df_model["default_flag"].notna().all()

In [10]:
# Show target counts and percentages in one table.
target_distribution = (
    df_model["default_flag"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Fully Paid", 1: "Charged Off"})
    .to_frame("loan_count")
)

target_distribution["share_pct"] = target_distribution["loan_count"] / len(df_model) * 100
target_distribution

,loan_count,share_pct
default_flag,,
Fully Paid,1499771,80.499549
Charged Off,363309,19.500451


**Target-construction finding.** The terminal-outcome sample contained **1,863,080 loans**: 1,499,771 fully paid and 363,309 charged off. The observed charge-off share was **19.50%**. This is the rate in the selected terminal-outcome sample—not the default rate for all loans originated—because active and unresolved statuses were excluded.

In [11]:
# Reassess missingness in the terminal-outcome sample.
model_missing_summary = (
    df_model.isna()
    .sum()
    .to_frame("missing_count")
)

model_missing_summary["missing_pct"] = model_missing_summary["missing_count"] / len(df_model) * 100
model_missing_summary.sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
emp_length,117230,6.292269
dti,1108,0.059471
annual_inc,4,0.000215
id,0,0.000000
int_rate,0,0.000000
funded_amnt,0,0.000000
grade,0,0.000000
loan_amnt,0,0.000000
home_ownership,0,0.000000
issue_d,0,0.000000


**Missingness after target filtering.** 117,230 rows (6.29%) had missing employment length, 1,108 had missing DTI, and four had missing annual income. The remaining selected fields had no missing values in this terminal-outcome sample. Employment length will use an explicit `Unknown` category; numerical missing values are retained for now.

In [12]:
# Standardize fields needed for analysis and the chronological split.
df_model["emp_length"] = df_model["emp_length"].fillna("Unknown")

# Convert percentage strings such as "10.65%" to numeric percentage points (10.65).
df_model["int_rate"] = (
    df_model["int_rate"]
    .astype("string")
    .str.replace("%", "", regex=False)
    .astype(float)
)

# Convert issue month strings such as "Dec-2011" to timestamps.
df_model["issue_d"] = pd.to_datetime(df_model["issue_d"], format="%b-%Y")

# Numerical missing values are intentionally not imputed here.
# Imputation will be fitted on development data only in Notebook 02.
df_model[["int_rate", "issue_d", "emp_length"]].dtypes

int_rate             float64
issue_d       datetime64[us]
emp_length               str
dtype: object

`int_rate` is now numeric but remains expressed in **percentage points** (for example, 10.65 means 10.65%). `issue_d` is a datetime field and can be used for a chronological split. Missing employment lengths are explicitly represented as `Unknown`; missing numeric values are left for the PD preprocessing pipeline so that imputation values can be learned from development data only.

Methodological note: In a production modeling pipeline, medians should be calculated on the training sample and then applied to validation data to prevent information leakage. Here, we're preparing the dataset; we'll implement that separation when we build the PD model.

In [13]:
# Screen for values that deserve review; do not delete records automatically.
invalid_value_counts = pd.Series({
    "loan_amnt <= 0": int((df_model["loan_amnt"] <= 0).sum()),
    "funded_amnt <= 0": int((df_model["funded_amnt"] <= 0).sum()),
    "annual_inc <= 0": int((df_model["annual_inc"] <= 0).sum()),
    "dti < 0": int((df_model["dti"] < 0).sum()),
    "int_rate <= 0": int((df_model["int_rate"] <= 0).sum()),
    "recoveries < 0": int((df_model["recoveries"] < 0).sum()),
    "fico_range_low > fico_range_high": int(
        (df_model["fico_range_low"] > df_model["fico_range_high"]).sum()
    ),
}, name="count").to_frame()

invalid_value_counts

,count
loan_amnt <= 0,0
funded_amnt <= 0,0
annual_inc <= 0,1085
dti < 0,2
int_rate <= 0,0
recoveries < 0,0
fico_range_low > fico_range_high,0


**Invalid-value review.** **1,085 loans had annual income at or below zero**. Such records cannot support a meaningful loan-to-income ratio, so that derived ratio will be set to missing for those records rather than to zero or infinity. The other checks are diagnostic: investigate non-zero counts before deciding on any treatment, and avoid removing observations without a defensible reason. `recoveries` is retained for later LGD analysis, not as a PD input.

## 5. Exploratory Credit-Risk Analysis

EDA is conducted on the terminal-outcome sample. The tables compare observed charge-off rates across key portfolio dimensions. These are descriptive associations, not causal effects or final estimates of future PD.

### 5.1 Target Distribution

The target distribution establishes the baseline event rate and shows how many observations belong to each outcome class.


In [14]:
# Count loans and calculate the percentage in each terminal-outcome class.
target_distribution

,loan_count,share_pct
default_flag,,
Fully Paid,1499771,80.499549
Charged Off,363309,19.500451


**Finding.** The terminal-outcome sample is imbalanced: approximately **80.50% are fully paid and 19.50% are charged off**. This baseline should be kept in view when evaluating the classifier; accuracy alone would not be a sufficient performance metric.

### 5.2 Default Rate by Credit Grade

LendingClub's assigned grade is compared with realized terminal outcomes. The table reports both the observed charge-off rate and the number of loans in each grade.

In [15]:
grade_default_rate = (
    df_model.groupby("grade")["default_flag"]
    .agg(default_rate="mean", loan_count="count")
)

# Convert the rate from a fraction to a percentage for easier interpretation.
grade_default_rate["default_rate"] *= 100
grade_default_rate.sort_index()

,default_rate,loan_count
grade,,
A,6.453820,346415
B,13.520441,546957
C,22.360119,528736
D,30.087199,275692
E,37.554257,117036
F,44.339446,37355
G,48.783176,10889


**Finding.** The observed charge-off rate rose consistently from **6.45% for Grade A** to **48.78% for Grade G**. The strong ordering suggests that credit grade captures meaningful risk segmentation in this sample. Grade is an origination-time lender assessment, so its use in the PD model should be interpreted as a predictor available within the lender's underwriting process—not as an independent borrower characteristic.

### 5.3 Default Rate by Annual Income

Annual income is divided into quartiles. The analysis compares observed charge-off rates across the income distribution; records without an income value cannot be assigned to a quartile and are excluded from this specific table.

In [16]:
# Use a temporary analysis frame so EDA-only bins are not saved as model features.
income_analysis = df_model.loc[
    df_model["annual_inc"].notna(), ["annual_inc", "default_flag"]
].copy()

income_analysis["income_quartile"] = pd.qcut(
    income_analysis["annual_inc"],
    q=4,
    labels=["Q1 - Lowest", "Q2", "Q3", "Q4 - Highest"],
    duplicates="drop",
)

income_default_rate = (
    income_analysis.groupby("income_quartile", observed=True)["default_flag"]
    .agg(default_rate="mean", loan_count="count")
)

income_default_rate["default_rate"] *= 100
income_default_rate

,default_rate,loan_count
income_quartile,,
Q1 - Lowest,22.485635,471968
Q2,20.512275,484037
Q3,18.724481,445497
Q4 - Highest,16.136091,461574


**Finding.** The observed charge-off rate decreased from **22.49% in the lowest-income quartile** to **16.14% in the highest-income quartile**, a difference of about 6.35 percentage points. This is consistent with income being associated with repayment capacity, but it does not establish causality or account for other borrower characteristics.

### 5.4 Default Rate by Loan Purpose

Loan purpose is compared to identify differences in observed charge-off rates across stated borrowing purposes. Loan counts are shown alongside rates because estimates based on smaller groups are less stable.

In [17]:

purpose_default_rate = (
    df_model.groupby("purpose")["default_flag"]
    .agg(default_rate="mean", loan_count="count")
    .sort_values("default_rate", ascending=False)
)

purpose_default_rate["default_rate"] *= 100

purpose_default_rate


,default_rate,loan_count
purpose,,
small_business,28.709615,20676
house,22.184331,11143
moving,22.122424,13249
renewable_energy,21.694378,1263
educational,20.754717,424
debt_consolidation,20.646859,1064559
medical,20.449726,22636
other,20.055608,114013
major_purchase,18.467748,41873


**Finding.** **Small-business loans had the highest observed charge-off rate (28.71%)** and **wedding loans the lowest (12.45%)**. Their sample sizes differ materially: for example, debt consolidation has over one million observations, while some purposes have only hundreds or a few thousand. Rate rankings for small categories should therefore be interpreted cautiously.


### 5.5 Default Rate by Loan Amount

Loan amount is examined to assess whether the size of the original loan is associated with differences in observed default risk.

Loans are grouped into quartiles based on loan amount. The default rate and number of loans are calculated for each group to compare risk across borrowing levels.


In [18]:
# Keep quartile labels in a temporary frame; they are for segmentation, not model inputs.
amount_analysis = df_model.loc[
    df_model["loan_amnt"].notna(), ["loan_amnt", "default_flag"]
].copy()

amount_analysis["loan_amount_quartile"] = pd.qcut(
    amount_analysis["loan_amnt"],
    q=4,
    labels=["Q1 - Lowest", "Q2", "Q3", "Q4 - Highest"],
    duplicates="drop",
)

loan_amount_default_rate = (
    amount_analysis.groupby("loan_amount_quartile", observed=True)["default_flag"]
    .agg(default_rate="mean", loan_count="count")
)

loan_amount_default_rate["default_rate"] *= 100
loan_amount_default_rate

,default_rate,loan_count
loan_amount_quartile,,
Q1 - Lowest,15.125957,466073
Q2,18.526010,477588
Q3,21.488527,503464
Q4 - Highest,23.114520,415955


**Finding.** The observed charge-off rate increased from **15.13% in the lowest loan-amount quartile** to **23.11% in the highest**, about 7.99 percentage points. Loan size is therefore a useful segmentation dimension to test in the PD model, although this univariate pattern may reflect other correlated factors.

## 6. Feature Engineering

Only a small number of interpretable derived variables are created. Exploratory quartile labels above remain in temporary analysis frames and are not added to the model dataset.

### 6.1 Average FICO Score

The dataset provides a lower and upper FICO range. Their midpoint is used as one compact numerical credit-score feature, avoiding the need to use both bounds as separate predictors.


In [19]:
# Use the midpoint of the reported FICO range as a single numeric feature.
df_model["fico_avg"] = (
    df_model["fico_range_low"] + df_model["fico_range_high"]
) / 2

df_model[["fico_range_low", "fico_range_high", "fico_avg"]].head()

,fico_range_low,fico_range_high,fico_avg
0,735.0,739.0,737.0
1,740.0,744.0,742.0
2,735.0,739.0,737.0
3,690.0,694.0,692.0
4,695.0,699.0,697.0


**Interpretation.** `fico_avg` provides a single summary of the reported FICO range. In Notebook 02, use either this derived score or the two original bounds—not all three together—unless there is a specific modelling reason to retain redundant predictors.

In [20]:
df_model["loan_to_income"] = np.where(
    df_model["annual_inc"] > 0,
    df_model["loan_amnt"] / df_model["annual_inc"],
    np.nan
)

### 6.2 Loan-to-Income Ratio

The loan-to-income ratio measures the original loan amount relative to reported annual income:

\[
\text{Loan-to-Income Ratio} =
\frac{\text{Loan Amount}}{\text{Annual Income}}
\]

It is calculated only when annual income is positive. Missing, zero, or negative income does not produce a meaningful ratio and is represented as missing.

In [21]:
# Avoid division by zero or negative income; leave those ratios missing.
positive_income = df_model["annual_inc"].gt(0)
df_model["loan_to_income"] = np.nan

df_model.loc[positive_income, "loan_to_income"] = (
    df_model.loc[positive_income, "loan_amnt"]
    / df_model.loc[positive_income, "annual_inc"]
)

# Review the distribution, including its upper tail.
df_model["loan_to_income"].describe(percentiles=[0.50, 0.95, 0.99])

count    1.861991e+06
mean     4.856340e-01
std      8.323137e+01
min      1.612903e-04
50%      1.982143e-01
95%      4.390244e-01
99%      5.769231e-01
max      4.000000e+04
Name: loan_to_income, dtype: float64

**Finding.** The median loan-to-income ratio was about **0.198**, while the maximum was **40,000**, showing an extremely right-skewed upper tail. This can arise when a positive reported income is very small relative to the loan amount. We retain the raw ratio here rather than arbitrarily cap observations; its treatment or suitability as a PD predictor should be assessed using development data in Notebook 02.

In [22]:
# Verify the number of ratios that cannot be calculated from the available income values.
df_model["loan_to_income"].isna().sum()

np.int64(1089)

The missing-ratio count includes records with missing annual income as well as records with non-positive income. These observations are not dropped automatically. If the ratio is used in the PD model, its missing-value treatment and any outlier transformation must be fitted using the development sample only.

In [23]:
# Review employment-length categories after representing missing values as "Unknown".
df_model["emp_length"].value_counts(dropna=False)

emp_length
10+ years    611800
2 years      168865
< 1 year     154172
3 years      149537
1 year       123272
Unknown      117230
5 years      116044
4 years      112416
6 years       85498
8 years       78753
7 years       78241
9 years       67252
Name: count, dtype: int64

**Employment-length review.** The observed categories are already consistently labelled (`< 1 year`, `1 year`, …, `10+ years`) and missing values are represented by `Unknown`. We leave the field categorical; encoding it numerically or with one-hot encoding belongs in the PD model pipeline.

## 7. Development and Validation Split

A chronological split is used so that development records precede validation records in time. The cutoff is selected at the 80th percentile of `issue_d`, with every loan in the cutoff month kept in the development sample. Because loans share monthly issue dates, the resulting row counts may not be exactly 80%/20%.


In [24]:
# Sort chronologically and choose the 80th-percentile issue date as the cutoff.
df_model = df_model.sort_values("issue_d").copy()
cutoff_date = df_model["issue_d"].quantile(0.80)

development_mask = df_model["issue_d"] <= cutoff_date
train_df = df_model.loc[development_mask].copy()
valid_df = df_model.loc[~development_mask].copy()

# Confirm that the two periods do not overlap in time.
assert train_df["issue_d"].max() < valid_df["issue_d"].min()

print("Cutoff date:", cutoff_date)
print("Development period:", train_df["issue_d"].min(), "to", train_df["issue_d"].max())
print("Validation period:", valid_df["issue_d"].min(), "to", valid_df["issue_d"].max())
print("Development loans:", f"{len(train_df):,}")
print("Validation loans:", f"{len(valid_df):,}")
print("Development share:", f"{len(train_df) / len(df_model):.2%}")
print("Validation share:", f"{len(valid_df) / len(df_model):.2%}")
print("Development default rate:", f"{train_df['default_flag'].mean():.2%}")
print("Validation default rate:", f"{valid_df['default_flag'].mean():.2%}")

Cutoff date: 2017-08-01 00:00:00
Development period: 2007-06-01 00:00:00 to 2017-08-01 00:00:00
Validation period: 2017-09-01 00:00:00 to 2020-09-01 00:00:00
Development loans: 1,495,018
Validation loans: 368,062
Development share: 80.24%
Validation share: 19.76%
Development default rate: 18.77%
Validation default rate: 22.47%


**Split finding.** The cutoff was **1 August 2017**: development covered June 2007–August 2017 and validation covered September 2017–September 2020. There were 1,495,018 development loans and 368,062 validation loans. The observed charge-off rate was **18.77% in development versus 22.47% in validation**, a 3.70 percentage-point increase. This temporal difference is important: the validation sample represents a higher-risk mix in this dataset and should not be replaced with a random split merely to equalize rates.

## 8. Save Prepared Datasets

The chronological development and validation samples are saved separately for the next notebooks. The CSVs retain the target and fields needed for model development and later simplified LGD/EAD analysis. Exploratory quartile labels are not included because they were created only for descriptive analysis.

Numerical imputation and categorical encoding have **not** been fitted here. Notebook 02 should fit its preprocessing steps using `train_df` only and apply the fitted transformations to `valid_df`. In particular, `recoveries` must not be included among PD predictors because it is observed after the outcome.

In [25]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

development_path = output_dir / "credit_risk_development.csv"
validation_path = output_dir / "credit_risk_validation.csv"

train_df.to_csv(development_path, index=False)
valid_df.to_csv(validation_path, index=False)

print(f"Development file: {development_path} ({len(train_df):,} rows)")
print(f"Validation file: {validation_path} ({len(valid_df):,} rows)")
print(f"Development file size: {development_path.stat().st_size / (1024**2):.1f} MB")
print(f"Validation file size: {validation_path.stat().st_size / (1024**2):.1f} MB")

Development file: ..\data\processed\credit_risk_development.csv (1,495,018 rows)
Validation file: ..\data\processed\credit_risk_validation.csv (368,062 rows)
Development file size: 201.6 MB
Validation file size: 49.8 MB


## 9. Key Findings and Limitations

### Key Findings

1. **Baseline outcome:** The terminal-outcome sample had an observed charge-off share of approximately 19.50%.
2. **Credit grade:** The observed rate rose from 6.45% for Grade A to 48.78% for Grade G.
3. **Borrower income:** The rate decreased from 22.49% in the lowest-income quartile to 16.14% in the highest-income quartile.
4. **Loan characteristics:** Small-business loans had the highest observed rate by purpose (28.71%); the rate also rose from 15.13% in the lowest loan-amount quartile to 23.11% in the highest.
5. **Temporal variation:** The validation period's observed rate was 22.47%, compared with 18.77% in the development period.

These are descriptive findings from this filtered sample. They help motivate candidate predictors but do not demonstrate causality or guarantee predictive power.

### Limitations and Handoff to Notebook 02

- **Terminal outcomes only:** active and unresolved loans are excluded. The 19.50% rate is therefore not a portfolio-wide rate across all loans in the source file.
- **Preprocessing discipline:** numeric missing values remain in the prepared files. Imputation, encoding, and any outlier treatment must be learned on development data only.
- **Target leakage:** `loan_status` is the source of `default_flag`, not a PD feature. `recoveries` is post-outcome information and must be excluded from PD predictors.
- **Simplified LGD/EAD coverage:** the selected fields include recoveries and funded amount but not the full payment history or principal outstanding at default. Later LGD/EAD estimates will require explicit simplifying assumptions.
- **Time period:** the data ends in 2020 Q3; patterns may not represent current lending conditions.

**Output:** `credit_risk_development.csv` and `credit_risk_validation.csv`, ready for the PD modelling pipeline in Notebook 02.